## Feature Engineering

In [1]:

import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
CLEAN_DIR = '../data/cleaned'
RAW_DIR   = '../data/raw'

In [4]:
df_users=pd.read_csv('../data/cleaned_data_/users_clean.csv')
df_bookings=pd.read_csv('../data/cleaned_data_/bookings_clean.csv')
df_snacks=pd.read_csv('../data/cleaned_data_/snacks_clean.csv')
df_reviews=pd.read_csv('../data/cleaned_data_/reviews_clean.csv')
df_chats=pd.read_csv('../data/cleaned_data_/chats_clean.csv')

In [5]:
# --- A: BOOKING FEATURES ---
print("\n⚙️ Extracting Booking Features...")
booking_features = df_bookings.groupby('user_id').agg(
    num_bookings       = ('id', 'count'),
    total_ticket_spend = ('total', 'sum'),
    avg_ticket_spend   = ('total', 'mean'),
    num_tickets_bought = ('tickets', 'sum'),
    fav_theater        = ('theater', lambda x: x.mode()[0] if len(x) > 0 else 'None'),
    fav_showtime       = ('showtime', lambda x: x.mode()[0] if len(x) > 0 else 'None')
).reset_index()


⚙️ Extracting Booking Features...


In [6]:
# --- B: CONCESSION / SNACK FEATURES ---
print("⚙️ Extracting Snack & Concession Features...")
snack_features = df_snacks.groupby('user_id').agg(
    num_snack_orders  = ('id', 'count'),
    total_snack_spend = ('total', 'sum'),
    avg_snack_spend   = ('total', 'mean')
).reset_index()

⚙️ Extracting Snack & Concession Features...


In [7]:

# --- C: REVIEW & SATISFACTION FEATURES ---
print("⚙️ Extracting Review & Feedback Features...")
review_features = df_reviews.groupby('user_id').agg(
    num_reviews = ('id', 'count'),
    avg_rating  = ('rating', 'mean')
).reset_index()


⚙️ Extracting Review & Feedback Features...


In [ ]:
# --- D: AI CHATBOT ENGAGEMENT FEATURES ---
print("⚙️ Extracting Chatbot Engagement Features...")
chat_features = df_chats[df_chats['user_id'] != 0].groupby('user_id').agg(
    num_chat_sessions = ('id', 'count')
).reset_index()

⚙️ Extracting Chatbot Engagement Features...


### Merging Into Master Feature Matrix

In [10]:
master = df_users[['id', 'username']].copy()
# Left-join all feature dataframes to preserve all registered users
master = master.merge(booking_features, left_on='id', right_on='user_id', how='left').drop(columns=['user_id'], errors='ignore')
master = master.merge(snack_features,   left_on='id', right_on='user_id', how='left').drop(columns=['user_id'], errors='ignore')
master = master.merge(review_features,  left_on='id', right_on='user_id', how='left').drop(columns=['user_id'], errors='ignore')
master = master.merge(chat_features,    left_on='id', right_on='user_id', how='left').drop(columns=['user_id'], errors='ignore')
# Handle missing values (users with 0 purchases or no reviews)
master = master.fillna({
    'num_bookings': 0,
    'total_ticket_spend': 0.0,
    'avg_ticket_spend': 0.0,
    'num_tickets_bought': 0,
    'fav_theater': 'None',
    'fav_showtime': 'None',
    'num_snack_orders': 0,
    'total_snack_spend': 0.0,
    'avg_snack_spend': 0.0,
    'num_reviews': 0,
    'avg_rating': 3.5,  # Neutral default rating for unrated users
    'num_chat_sessions': 0
})

### Categorial Encoding

In [11]:
le_theater  = LabelEncoder()
le_showtime = LabelEncoder()
master['fav_theater_enc']  = le_theater.fit_transform(master['fav_theater'].astype(str))
master['fav_showtime_enc'] = le_showtime.fit_transform(master['fav_showtime'].astype(str))
# Round float columns for clean formatting
float_cols = ['total_ticket_spend', 'avg_ticket_spend', 'total_snack_spend', 'avg_snack_spend', 'avg_rating']
master[float_cols] = master[float_cols].round(2)

In [12]:
os.makedirs(CLEAN_DIR, exist_ok=True)
output_file = os.path.join(CLEAN_DIR, 'master_feature_matrix.csv')
master.to_csv(output_file, index=False)
print("\n" + "=" * 55)
print("🎉 PHASE 3 FEATURE ENGINEERING COMPLETE")
print("=" * 55)
print(f"• Master Matrix File  : {output_file}")
print(f"• Total Rows (Users)  : {master.shape[0]}")
print(f"• Total Columns       : {master.shape[1]}")
print("\nFeature Columns Created:")
for col in master.columns:
    print(f"  └─ {col:22s} ({master[col].dtype})")
print("\nFirst 3 Customer Vector Samples:")
print(master[['username', 'num_bookings', 'total_ticket_spend', 'total_snack_spend', 'num_chat_sessions', 'avg_rating']].head(3).to_string(index=False))
print("=" * 55)


🎉 PHASE 3 FEATURE ENGINEERING COMPLETE
• Master Matrix File  : ../data/cleaned\master_feature_matrix.csv
• Total Rows (Users)  : 45
• Total Columns       : 16

Feature Columns Created:
  └─ id                     (int64)
  └─ username               (object)
  └─ num_bookings           (int64)
  └─ total_ticket_spend     (float64)
  └─ avg_ticket_spend       (float64)
  └─ num_tickets_bought     (int64)
  └─ fav_theater            (object)
  └─ fav_showtime           (object)
  └─ num_snack_orders       (float64)
  └─ total_snack_spend      (float64)
  └─ avg_snack_spend        (float64)
  └─ num_reviews            (float64)
  └─ avg_rating             (float64)
  └─ num_chat_sessions      (int64)
  └─ fav_theater_enc        (int64)
  └─ fav_showtime_enc       (int64)

First 3 Customer Vector Samples:
    username  num_bookings  total_ticket_spend  total_snack_spend  num_chat_sessions  avg_rating
sarah_connor             9               420.5               88.5                  3      